# TechDistribuidora con Apache Camel y Google Drive
**Integración de Sistemas | Johao Gavilanes y Erick Granda | Ingeniería de Software**

Este notebook ejecuta Apache Camel en Java dentro de Colab. Python prepara el entorno y muestra los resultados; la transferencia la hace la ruta `transferencia-pedidos-csv`.

## Antes de ejecutar
1. Johao crea en **Mi unidad** la carpeta `TechDistribuidora` y la comparte con Erick como editor.
2. Dentro crea `input`, `output` y `logs`. Erick encuentra la carpeta en Compartido conmigo.
3. Si Erick ejecuta este notebook con su cuenta, agrega un acceso directo de esa carpeta a **Mi unidad** y verifica su ruta en el panel Archivos de Colab. Solo **un notebook destino** debe estar activo.
4. Sistema origen: Erick sube `ejemplos/pedidos_001.csv` a `input`. Sistema destino: Johao ejecuta este notebook y muestra `output` y `logs`.
5. Ejecuta las celdas en orden. No subas los ejemplos a input antes de haber iniciado Camel si quieres demostrar detección de un archivo nuevo.

Colab descarga Maven y las bibliotecas al entorno temporal remoto. No instala programas en tu computadora. La primera compilación necesita Internet y puede tardar varios minutos.


## 1 Conectar Google Drive
Autoriza tu propia cuenta de Google cuando Colab lo solicite. El montaje da acceso al Drive de quien ejecuta la celda; compartir el notebook no comparte automáticamente la carpeta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 Elegir la carpeta compartida
La carpeta base debe existir en Drive. Si no aparece, verifica el acceso directo y copia su ruta desde el panel Archivos. `VALIDAR_PEDIDOS=False` cumple la transferencia básica de cualquier CSV. Después usa `True` para la ampliación: cabeceras, campos requeridos, cantidad entera positiva y precio positivo.

In [ ]:
from pathlib import Path
BASE = Path('/content/drive/MyDrive/TechDistribuidora')
VALIDAR_PEDIDOS = False
if not BASE.is_dir():
    raise FileNotFoundError(f'No existe {BASE}. Crea o ubica la carpeta compartida antes de continuar.')
INPUT, OUTPUT, LOGS = [BASE / name for name in ('input', 'output', 'logs')]
for folder in (INPUT, OUTPUT, LOGS):
    folder.mkdir(exist_ok=True)
print('Entrada:', INPUT, '\nSalida:', OUTPUT, '\nLogs:', LOGS)


## 3 Preparar Camel en el entorno remoto
La aplicación usa Java 17, 21 o 25. Colab normalmente incluye Java; esta celda lo comprueba y deja un mensaje si el entorno cambió. Descarga Maven 3.9.9 de Maven Central y verifica SHA-512. No usa `pip` ni ejecuta código remoto desconocido.

In [ ]:
import hashlib, re, shutil, subprocess, urllib.request, zipfile
TOOLS = Path('/content/tech-tools')
TOOLS.mkdir(exist_ok=True)
if not shutil.which('java'):
    raise RuntimeError('Este entorno no tiene Java. Selecciona un entorno Colab con Java 17, 21 o 25.')
version = subprocess.run(['java', '-version'], capture_output=True, text=True)
print(version.stderr)
match = re.search(r'version "(\d+)', version.stderr)
if not match or int(match.group(1)) not in (17, 21, 25):
    raise RuntimeError('Usa Java 17, 21 o 25, versiones compatibles con Camel 4.22.0.')
url = 'https://repo.maven.apache.org/maven2/org/apache/maven/apache-maven/3.9.9/apache-maven-3.9.9-bin.zip'
archive = TOOLS / 'maven.zip'
MAVEN = TOOLS / 'apache-maven-3.9.9/bin/mvn'
if not MAVEN.exists():
    urllib.request.urlretrieve(url, archive)
    expected = urllib.request.urlopen(url + '.sha512').read().decode().strip().split()[0]
    assert hashlib.sha512(archive.read_bytes()).hexdigest() == expected, 'Descarga dañada'
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(TOOLS)
    MAVEN.chmod(0o755)
print('Maven preparado:', MAVEN)


## 4 Crear y compilar el proyecto
El código incluido es el mismo del repositorio. `App` configura las rutas y el historial; `FileTransferRoute` detecta CSV, valida opcionalmente, copia y registra el resultado. Abre esos archivos para explicarlos durante la presentación.

La celda termina con `BUILD SUCCESS`. Si falla, muestra las últimas líneas del error y conserva el log completo en `/content/techdistribuidora/build.log`.

In [ ]:
SOURCES = {'pom.xml': '<?xml version="1.0" encoding="UTF-8"?>\n<project xmlns="http://maven.apache.org/POM/4.0.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">\n  <modelVersion>4.0.0</modelVersion>\n  <groupId>ec.edu.udla</groupId>\n  <artifactId>techdistribuidora</artifactId>\n  <version>1.0.0</version>\n  <properties>\n    <maven.compiler.release>17</maven.compiler.release>\n    <project.build.sourceEncoding>UTF-8</project.build.sourceEncoding>\n    <camel.version>4.22.0</camel.version>\n  </properties>\n  <dependencyManagement><dependencies><dependency>\n    <groupId>org.apache.camel</groupId><artifactId>camel-bom</artifactId><version>${camel.version}</version><type>pom</type><scope>import</scope>\n  </dependency></dependencies></dependencyManagement>\n  <dependencies>\n    <dependency><groupId>org.apache.camel</groupId><artifactId>camel-main</artifactId></dependency>\n    <dependency><groupId>org.apache.camel</groupId><artifactId>camel-file</artifactId></dependency>\n    <dependency><groupId>org.apache.camel</groupId><artifactId>camel-core-languages</artifactId></dependency>\n    <dependency><groupId>org.apache.commons</groupId><artifactId>commons-csv</artifactId><version>1.14.1</version></dependency>\n    <dependency><groupId>org.slf4j</groupId><artifactId>slf4j-simple</artifactId><version>2.0.17</version></dependency>\n  </dependencies>\n  <build><plugins>\n    <plugin><groupId>org.apache.maven.plugins</groupId><artifactId>maven-compiler-plugin</artifactId><version>3.14.1</version></plugin>\n    <plugin><groupId>org.apache.maven.plugins</groupId><artifactId>maven-shade-plugin</artifactId><version>3.6.1</version>\n      <executions><execution><phase>package</phase><goals><goal>shade</goal></goals><configuration>\n        <createDependencyReducedPom>false</createDependencyReducedPom>\n        <filters><filter><artifact>*:*</artifact><excludes><exclude>META-INF/*.SF</exclude><exclude>META-INF/*.DSA</exclude><exclude>META-INF/*.RSA</exclude></excludes></filter></filters>\n        <transformers>\n          <transformer implementation="org.apache.maven.plugins.shade.resource.ServicesResourceTransformer"/>\n          <transformer implementation="org.apache.maven.plugins.shade.resource.ManifestResourceTransformer"><mainClass>ec.edu.udla.App</mainClass></transformer>\n        </transformers>\n      </configuration></execution></executions>\n    </plugin>\n  </plugins></build>\n</project>\n', 'src/main/java/ec/edu/udla/App.java': 'package ec.edu.udla;\n\nimport java.nio.file.Files;\nimport java.nio.file.Path;\nimport java.util.HashMap;\nimport java.util.Map;\nimport org.apache.camel.main.Main;\nimport org.apache.camel.support.processor.idempotent.FileIdempotentRepository;\n\n/** Aplicación standalone. Python en Colab solo prepara y controla este proceso. */\npublic final class App {\n    public static void main(String[] args) throws Exception {\n        Map<String, String> options = new HashMap<>();\n        for (String arg : args) {\n            if (!arg.startsWith("--") || !arg.contains("=")) {\n                throw new IllegalArgumentException("Use --input=RUTA --output=RUTA --logs=RUTA --validate=true|false");\n            }\n            String[] pair = arg.substring(2).split("=", 2);\n            options.put(pair[0], pair[1]);\n        }\n        Path input = Path.of(options.getOrDefault("input", "input")).toAbsolutePath().normalize();\n        Path output = Path.of(options.getOrDefault("output", "output")).toAbsolutePath().normalize();\n        Path logs = Path.of(options.getOrDefault("logs", "logs")).toAbsolutePath().normalize();\n        if (input.equals(output) || output.startsWith(input) || input.startsWith(output)) {\n            throw new IllegalArgumentException("input y output deben ser carpetas diferentes sin anidamiento");\n        }\n        for (Path dir : new Path[]{input, output, logs}) Files.createDirectories(dir);\n        boolean validate = Boolean.parseBoolean(options.getOrDefault("validate", "false"));\n        Main main = new Main();\n        main.bind("processedFiles", FileIdempotentRepository.fileIdempotentRepository(\n            logs.resolve("processed-files.dat").toFile(), 10000));\n        main.configure().addRoutesBuilder(new FileTransferRoute(input, output, logs, validate));\n        System.out.println("INPUT=" + input + " OUTPUT=" + output + " VALIDATE=" + validate);\n        main.run();\n    }\n}\n', 'src/main/java/ec/edu/udla/FileTransferRoute.java': 'package ec.edu.udla;\n\nimport java.io.StringReader;\nimport java.math.BigDecimal;\nimport java.nio.charset.StandardCharsets;\nimport java.nio.file.Files;\nimport java.nio.file.Path;\nimport java.nio.file.StandardOpenOption;\nimport java.time.OffsetDateTime;\nimport java.time.ZoneOffset;\nimport java.util.Set;\nimport org.apache.camel.Exchange;\nimport org.apache.camel.LoggingLevel;\nimport org.apache.camel.builder.RouteBuilder;\nimport org.apache.commons.csv.CSVFormat;\nimport org.apache.commons.csv.CSVParser;\nimport org.apache.commons.csv.CSVRecord;\n\n/** File Transfer con conservación del original, filtro e historial persistente. */\npublic final class FileTransferRoute extends RouteBuilder {\n    private final Path input, output, logs;\n    private final boolean validate;\n    public FileTransferRoute(Path input, Path output, Path logs, boolean validate) {\n        this.input = input; this.output = output; this.logs = logs; this.validate = validate;\n    }\n    private static String uriPath(Path path) { return path.toString().replace(\'\\\\\', \'/\'); }\n    @Override public void configure() {\n        // CSV inválido: registrar rechazo y conservar el original para su corrección.\n        onException(IllegalArgumentException.class).maximumRedeliveries(0).handled(true)\n            .process(e -> {\n                Throwable error = e.getProperty(Exchange.EXCEPTION_CAUGHT, Throwable.class);\n                audit(e, "ERROR_VALIDACION", error.getMessage(), logs.resolve("errores.csv"));\n            })\n            .log(LoggingLevel.WARN, "Rechazado ${header.CamelFileName}: ${exception.message}");\n        // Fallo técnico: reintentar, registrar y dejar el intercambio fallido.\n        onException(Exception.class).maximumRedeliveries(3).redeliveryDelay(1000).handled(false)\n            .process(e -> {\n                Throwable error = e.getProperty(Exchange.EXCEPTION_CAUGHT, Throwable.class);\n                audit(e, "ERROR_TECNICO", error.toString(), logs.resolve("errores.csv"));\n            });\n        from("file:" + uriPath(input)\n            + "?include=.*[.]csv&noop=true&delay=2000&initialDelay=0"\n            + "&readLock=changed&readLockCheckInterval=1000&readLockTimeout=10000&readLockMinLength=0"\n            + "&idempotentRepository=#processedFiles"\n            + "&idempotentKey=${file:absolute.path}-${file:size}-${file:modified}")\n            .routeId("transferencia-pedidos-csv")\n            .process(e -> { if (validate) validateOrders(e); })\n            .to("file:" + uriPath(output) + "?fileExist=Override&tempFileName=.${file:name}.part")\n            .process(e -> audit(e, "COPIADO", "Archivo transferido sin transformar su contenido", logs.resolve("transferencias.csv")))\n            .log("COPIADO ${header.CamelFileName} -> output");\n    }\n    private void validateOrders(Exchange e) throws Exception {\n        // Se valida una representación del texto; el cuerpo original no se reemplaza.\n        String text = e.getMessage().getBody(String.class);\n        if (text.startsWith("\\uFEFF")) text = text.substring(1);\n        CSVFormat format = CSVFormat.DEFAULT.builder().setHeader().setSkipHeaderRecord(true).get();\n        try (CSVParser parser = format.parse(new StringReader(text))) {\n            Set<String> expected = Set.of("pedido_id", "cliente", "producto", "cantidad", "precio_unitario");\n            if (!parser.getHeaderMap().keySet().equals(expected) || parser.getHeaderNames().size() != 5)\n                throw new IllegalArgumentException("Cabecera requerida: pedido_id,cliente,producto,cantidad,precio_unitario");\n            int count = 0;\n            for (CSVRecord row : parser) {\n                count++;\n                if (!row.isConsistent()) throw new IllegalArgumentException("Fila " + count + ": número de columnas incorrecto");\n                for (String field : expected) if (row.get(field).isBlank())\n                    throw new IllegalArgumentException("Fila " + count + ": campo vacío " + field);\n                try {\n                    if (Integer.parseInt(row.get("cantidad")) <= 0 || new BigDecimal(row.get("precio_unitario")).signum() <= 0)\n                        throw new NumberFormatException();\n                } catch (NumberFormatException ex) {\n                    throw new IllegalArgumentException("Fila " + count + ": cantidad entera y precio deben ser positivos");\n                }\n            }\n            if (count == 0) throw new IllegalArgumentException("CSV sin pedidos");\n        } catch (java.io.UncheckedIOException | java.io.IOException ex) {\n            throw new IllegalArgumentException("CSV no legible: " + ex.getMessage(), ex);\n        }\n    }\n    private static String quote(Object value) {\n        return "\\"" + String.valueOf(value).replace("\\"", "\\"\\"").replace(\'\\n\', \' \').replace(\'\\r\', \' \') + "\\"";\n    }\n    private void audit(Exchange e, String status, String detail, Path file) throws Exception {\n        if (!Files.exists(file)) Files.writeString(file, "fecha,archivo,estado,detalle\\n", StandardCharsets.UTF_8, StandardOpenOption.CREATE_NEW);\n        String line = quote(OffsetDateTime.now(ZoneOffset.ofHours(-5))) + ","\n            + quote(e.getMessage().getHeader(Exchange.FILE_NAME)) + "," + quote(status) + "," + quote(detail) + "\\n";\n        Files.writeString(file, line, StandardCharsets.UTF_8, StandardOpenOption.APPEND);\n    }\n}\n'}
APP = Path('/content/techdistribuidora')
APP.mkdir(exist_ok=True)
for relative, contents in SOURCES.items():
    destination = APP / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding='utf-8')
with (APP / 'build.log').open('w') as build_log:
    result = subprocess.run([str(MAVEN), '--batch-mode', '-Dmaven.repo.local=/content/tech-tools/m2', 'package'],
                            cwd=APP, stdout=build_log, stderr=subprocess.STDOUT)
tail = (APP / 'build.log').read_text(errors='replace').splitlines()[-25:]
print('\n'.join(tail))
if result.returncode:
    raise RuntimeError('La compilación falló. Revisa build.log antes de continuar.')
JAR = APP / 'target/techdistribuidora-1.0.0.jar'
assert JAR.exists(), 'No se generó el ejecutable'


## 5 Iniciar el sistema destino
Solo inicia una instancia por carpeta. La aplicación revisa input cada dos segundos, conserva originales y mantiene un historial persistente en logs. Subir un archivo modificado lo identifica como una nueva versión mediante ruta, tamaño y fecha de modificación.

Con el mismo nombre se reemplaza la copia de output. Utiliza nombres únicos, por ejemplo `pedidos_erick_001.csv`, para evitar confusión. El historial es una ayuda de la demo, no una garantía transaccional de exactamente una entrega.

In [ ]:
import time
if 'PROC' in globals() and PROC.poll() is None:
    raise RuntimeError('Camel ya está activo. Detén la instancia antes de iniciar otra.')
CONSOLE = LOGS / 'camel-console.log'
console_handle = CONSOLE.open('a', encoding='utf-8')
PROC = subprocess.Popen(['java', '-jar', str(JAR), f'--input={INPUT}', f'--output={OUTPUT}',
                         f'--logs={LOGS}', f'--validate={str(VALIDAR_PEDIDOS).lower()}'],
                        stdout=console_handle, stderr=subprocess.STDOUT)
time.sleep(3)
if PROC.poll() is not None:
    console_handle.close()
    print(CONSOLE.read_text(errors='replace')[-6000:])
    raise RuntimeError('Camel no arrancó. Revisa el log mostrado.')
print('Camel activo. Erick: sube ahora pedidos_001.csv y no_procesar.txt a input.')
print(CONSOLE.read_text(errors='replace')[-3000:])


## 6 Monitorear durante cinco minutos
Mientras se ejecuta esta celda, el origen sube el archivo desde otra cuenta. Verás `COPIADO ... -> output`. El montaje de Drive puede reflejar cambios con demora; si no aparece, revisa ruta, permisos y tamaño estable del archivo. Puedes volver a ejecutar la celda para otra ventana de cinco minutos.

No cierres la sesión: Colab no es un servicio de monitoreo permanente. Detener esta celda interrumpe la visualización; usa la celda 8 para apagar Camel.

In [ ]:
DURACION_SEGUNDOS = 300
position = 0
start = time.monotonic()
try:
    while time.monotonic() - start < DURACION_SEGUNDOS:
        if PROC.poll() is not None:
            raise RuntimeError('El proceso Camel terminó. Revisa camel-console.log.')
        with CONSOLE.open(encoding='utf-8', errors='replace') as stream:
            stream.seek(position)
            new_text = stream.read()
            position = stream.tell()
        if new_text:
            print(new_text, end='')
        time.sleep(2)
except KeyboardInterrupt:
    print('Visualización detenida. Camel sigue activo hasta ejecutar la celda 8.')
print('Archivos de salida:', [p.name for p in OUTPUT.iterdir() if p.is_file()])


## 7 Verificar contenido y consultar los logs
Esta celda compara SHA-256 de cada copia con su original. La igualdad prueba que no se transformó el contenido. Un archivo TXT no debe estar en output; no tiene una fila de transferencia porque el filtro lo ignora.

`transferencias.csv` registra copias exitosas. `errores.csv` aparece cuando ocurre un rechazo o fallo técnico. No elimines el historial para preparar capturas: utiliza archivos con nombres nuevos.

In [ ]:
import csv
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(65536), b''):
            digest.update(block)
    return digest.hexdigest()
for copy in sorted(OUTPUT.iterdir()):
    if copy.is_file() and copy.suffix.lower() == '.csv':
        original = INPUT / copy.name
        equal = original.exists() and sha256(original) == sha256(copy)
        print(copy.name, '| original conservado:', original.exists(), '| contenido idéntico:', equal)
for filename in ('transferencias.csv', 'errores.csv'):
    path = LOGS / filename
    print('\n' + filename)
    if path.exists():
        with path.open(encoding='utf-8', newline='') as stream:
            for row in csv.DictReader(stream):
                print(row)
    else:
        print('Todavía no hay eventos de este tipo.')


## 8 Detener Camel y activar la ampliación
Detén Camel antes de cambiar `VALIDAR_PEDIDOS`. Para demostrar la validación: detén, cambia a `True` en la celda 2, vuelve a ejecutar 2 y 5, y sube `pedidos_invalidos.csv` con un nombre que no se haya procesado. Consulta las celdas 6 y 7: debe aparecer `ERROR_VALIDACION` y no debe existir copia de ese archivo.

Si corriges el CSV y vuelves a subirlo, su modificación permite procesar la nueva versión. Conserva ambas capturas para explicar el error y la corrección.

In [ ]:
if 'PROC' in globals() and PROC.poll() is None:
    PROC.terminate()
    try:
        PROC.wait(timeout=20)
    except subprocess.TimeoutExpired:
        PROC.kill()
        PROC.wait(timeout=10)
if 'console_handle' in globals() and not console_handle.closed:
    console_handle.close()
print('Camel detenido.')


## 9 Evidencias de la pareja y entrega
- Captura E1: carpeta compartida y permisos de ambos integrantes, sin exponer otros archivos personales.
- Captura E2: origen sube un CSV nuevo a input desde su cuenta.
- Captura E3: Camel arrancado y línea COPIADO en Colab.
- Captura E4: original en input y copia en output.
- Captura E5: comparación de contenido y fila de transferencias.csv.
- Captura E6: TXT excluido.
- Captura E7: validación activada, CSV rechazado y fila de errores.csv.

Guarda las capturas en una carpeta `evidencias` de la entrega y pégalas en el informe. No confundas las pruebas locales suministradas con la ejecución compartida de Drive. Completa docente, curso, fecha real de ejecución y vínculos de carpeta/notebook.

Repositorio: https://github.com/JohaoG10/Tech-DIstribuidora-Taller

La bitácora incluye los prompts utilizados en esta conversación y un espacio para los que añadan durante la práctica. Cada integrante debe poder explicar la ruta Java y las celdas de preparación.